# GemCol Evaluation: Phase 4 (Reciprocal Rank Fusion)
This notebook merges the retrieval results from the sparse BM25 baseline and the dense BGE baseline using RRF. RRF combines the two models without needing to normalize their wildly different score distributions.

In [ ]:
import sys
import json

sys.path.insert(0, '/workspace/gemcol_evaluation')
from utils import load_msmarco_dev, save_run_json, evaluate_run, print_metrics, ExperimentTracker
from utils.retrieval import rrf_fusion

print("Loading ground truth qrels...")
_, qrels, _ = load_msmarco_dev()

def load_run(path):
    with open(path, 'r') as f:
        return json.load(f)

print("Loading BM25 and BGE runs...")
bm25_run = load_run("/workspace/results/bm25_run.json")
bge_run = load_run("/workspace/results/bge_run.json")

In [ ]:
print("Applying Reciprocal Rank Fusion (k=60, top_k=100)...")
rrf_run = rrf_fusion([bm25_run, bge_run], k=60, top_k=100)

save_run_json(rrf_run, "/workspace/results/rrf_run.json")
print("✅ RRF fusion complete and saved.")

In [ ]:
metrics = evaluate_run(rrf_run, qrels)
print_metrics(metrics, system_name="RRF Fusion (BM25 + BGE)")

tracker = ExperimentTracker("/workspace/results/experiments.json")
tracker.log("RRF Fusion", config={"models": ["bm25s", "BAAI/bge-large-en-v1.5"], "fusion": "rrf", "k": 60}, metrics={
    "ndcg@10": metrics["ndcg@10"],
    "mrr@10": metrics["mrr@10"],
    "recall@100": metrics["recall@100"]
})
print("✅ Metrics logged to experiments.json")